In [16]:
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path

In [17]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Exploring games data

In [18]:
clean_games_fp = Path(config['games_folder'] + config['games_clean_fp'])

games_df = pd.read_json(clean_games_fp, orient='records')
games_df['updated_at'] = pd.to_datetime(games_df['updated_at'], errors='coerce')
games_df['first_release_date'] = pd.to_datetime(games_df['first_release_date'], errors='coerce')

games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,Pac-Man 99,78.618734,2026-04-28 23:14:48,2021-04-07,145515,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,PUBG: Battlegrounds,71.385811,2026-04-28 23:13:58,2017-12-20,27789,1,1,0,1,1,...,0,0,0,1,1,0,0,0,0,0
2,LEGO Star Wars III: The Clone Wars,73.796274,2026-04-28 23:12:58,2011-02-28,6844,0,1,0,1,1,...,0,0,1,0,0,0,0,0,0,0
3,Team Fortress 2,82.886431,2026-04-28 23:11:52,2007-10-10,891,0,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
4,Ultimate Chicken Horse,78.068396,2026-04-28 23:11:13,2016-03-04,18158,0,1,0,1,1,...,0,0,0,1,0,0,0,0,1,0


In [19]:
games_df.head()

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,Pac-Man 99,78.618734,2026-04-28 23:14:48,2021-04-07,145515,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
1,PUBG: Battlegrounds,71.385811,2026-04-28 23:13:58,2017-12-20,27789,1,1,0,1,1,...,0,0,0,1,1,0,0,0,0,0
2,LEGO Star Wars III: The Clone Wars,73.796274,2026-04-28 23:12:58,2011-02-28,6844,0,1,0,1,1,...,0,0,1,0,0,0,0,0,0,0
3,Team Fortress 2,82.886431,2026-04-28 23:11:52,2007-10-10,891,0,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
4,Ultimate Chicken Horse,78.068396,2026-04-28 23:11:13,2016-03-04,18158,0,1,0,1,1,...,0,0,0,1,0,0,0,0,1,0


In [20]:
#Checking to make sure the id uniquely identifies each row
print("No duplicate IDS") if len(games_df['id'].unique()) == len(games_df) else print("Duplicate IDs found")

No duplicate IDS


In [21]:
#Checking to see if duplicate game names exist
print("No duplicate game names") if len(games_df['name'].unique()) == len(games_df) else print("Duplicate game names found")

Duplicate game names found


In [22]:
#Get all rows with duplicate game names
duplicate_names_df = games_df[games_df.duplicated(subset=['name'], keep=False)].sort_values('name')
duplicate_names_df.head(6)

,name,rating,updated_at,first_release_date,id,game_modes_Battle Royale,game_modes_Co-operative,game_modes_Massively Multiplayer Online (MMO),game_modes_Multiplayer,game_modes_Single player,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
17005,10-Pin Bowling,-1.000000,2024-11-14 09:33:09,1984-12-31,153453,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
17006,10-Pin Bowling,-1.000000,2024-11-14 09:27:27,1999-08-01,92273,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
7059,15 Minutes,-1.000000,2026-04-09 19:38:22,2026-01-05,395433,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
12896,15 Minutes,-1.000000,2026-02-02 16:52:35,2025-10-23,355071,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5508,1942,61.392350,2026-04-18 08:37:59,1985-12-11,272544,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
6432,1942,67.742592,2026-04-13 08:03:19,1984-12-01,6075,0,1,0,0,1,...,0,0,0,0,0,1,0,0,0,0


In [23]:
#Find columns where duplicates differ (excluding name, rating, updated_at, id)
exclude_cols = {'name', 'rating', 'updated_at', 'id'}
cols_to_check = [col for col in duplicate_names_df.columns if col not in exclude_cols]

i = 0
for game_name in duplicate_names_df['name'].unique():
    game_group = duplicate_names_df[duplicate_names_df['name'] == game_name]
    differing_cols = []
    for col in cols_to_check:
        if len(game_group[col].unique()) > 1:
            differing_cols.append(col)
    if'first_release_date' not in differing_cols:  # Print every 10th game to avoid too much output
        #print(f"{game_name}: {differing_cols}")
        pass
    i+=1

There are many duplicates but they may functionally differ so we'll keep them for now

# Exploring multiplayer modes data

In [24]:
clean_modes_fp = Path(config['multiplayer_modes_folder'] + config['multiplayer_modes_clean_fp'])

modes_df = pd.read_json(clean_modes_fp, orient='records')

print("Contains " + str(len(modes_df)) + " rows")
modes_df.head()

Contains 24094 rows


,id,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen
0,9953,92273,False,False,False,-1,2,False,-1,-1,Game Boy Color,False
1,1832,7153,True,True,True,2,0,False,0,0,Xbox 360,False
2,7987,57887,False,False,True,0,2,False,0,0,Xbox One,False
3,7,46076,False,False,False,-1,30,False,-1,-1,PC (Microsoft Windows),False
4,10207,31256,False,False,False,-1,-1,False,-1,-1,Web browser,False


In [25]:
modes_df.isna().mean().sort_values(ascending=False)

id                0.0
game              0.0
dropin            0.0
campaigncoop      0.0
offlinecoop       0.0
offlinecoopmax    0.0
offlinemax        0.0
onlinecoop        0.0
onlinecoopmax     0.0
onlinemax         0.0
platform          0.0
splitscreen       0.0
dtype: float64

-1 as a fill value works fine for all of these

In [26]:
def isunique(df, subset):
    return len(df[subset].unique()) == len(df)

print("No duplicate IDS") if isunique(modes_df, 'id') else print("Duplicate IDs found")
print("No duplicate game ids") if isunique(modes_df, 'game') else print("Duplicate game ids found")

No duplicate IDS
Duplicate game ids found


In [27]:
duplicate_game_ids_df = modes_df[modes_df.duplicated(subset=['game'], keep=False)].sort_values('game')
duplicate_game_ids_df = duplicate_game_ids_df.merge(games_df[['id', 'name']], left_on='game', right_on='id', how='left', suffixes=('_mode', '_game'))
duplicate_game_ids_df.head(10)

,id_mode,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,platform,splitscreen,id_game,name
0,11593,72,False,False,True,2,0,True,2,0,PC (Microsoft Windows),True,72.0,Portal 2
1,11592,72,False,False,True,2,0,True,2,0,PlayStation 3,True,72.0,Portal 2
2,11591,72,False,False,True,2,0,True,2,0,Xbox 360,True,72.0,Portal 2
3,11595,72,False,False,True,2,0,True,2,0,Linux,True,72.0,Portal 2
4,11594,72,False,False,True,2,0,True,2,0,Mac,True,72.0,Portal 2
5,17435,83,False,False,False,2,-1,False,-1,-1,PC (Microsoft Windows),True,83.0,Baldur's Gate: Dark Alliance
6,1631,83,False,True,True,2,0,False,0,0,PlayStation 2,False,83.0,Baldur's Gate: Dark Alliance
7,11138,121,True,True,False,0,0,False,0,0,Mac,False,121.0,Minecraft: Java Edition
8,11137,121,True,True,False,0,0,False,0,0,PC (Microsoft Windows),False,121.0,Minecraft: Java Edition
9,11139,121,True,True,False,0,0,False,0,0,Linux,False,121.0,Minecraft: Java Edition


Duplicate IDS often seems to indicate multi-platform releases

In [ ]:
full_df = modes_df.merge(games_df, left_on='game', right_on='id', how='left', suffixes=('', '_game'))
full_df.head()

,id_mode,game,dropin,campaigncoop,offlinecoop,offlinecoopmax,offlinemax,onlinecoop,onlinecoopmax,onlinemax,...,platforms_WonderSwan Color,platforms_Xbox,platforms_Xbox 360,platforms_Xbox One,platforms_Xbox Series X|S,platforms_ZX Spectrum,platforms_Zeebo,platforms_e-Reader / Card-e Reader,platforms_iOS,platforms_visionOS
0,9953,92273,False,False,False,-1,2,False,-1,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1832,7153,True,True,True,2,0,False,0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,7987,57887,False,False,True,0,2,False,0,0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,7,46076,False,False,False,-1,30,False,-1,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,10207,31256,False,False,False,-1,-1,False,-1,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Handling conflicting values across columns

In [44]:
columns = ['onlinecoop', 'onlinecoopmax', 'onlinemax']

def cap_at_1(x):
    return 1 if x >= 1 else x

for col in columns[1:]:
    full_df[col] = full_df[col].apply(cap_at_1)

full_df[columns[-1]].value_counts()

onlinemax
-1    15037
 1     6572
 0     2485
Name: count, dtype: int64

In [45]:
#Sample one row from each unique combination of columns
selected_cols = ['name', 'first_release_date', 'platform'] + [col for col in full_df.columns if 'game_modes' in col]
conflict_scenarios_df = full_df.groupby(columns)[selected_cols].agg(lambda x: x.sample(1))

#conflict_scenarios_df = conflict_scenarios_df.drop(labels = [()])
conflict_scenarios_df

name  \
onlinecoop onlinecoopmax onlinemax                                                      
False      -1            -1                                             Badass Tennis   
                          0                                                       Uno   
                          1                                                    Doom 3   
            0            -1                             The Last One and Then Another   
                          0         The King of Fighters: Maximum Impact Regulation A   
                          1                                      Magical Battle Arena   
            1            -1                      Risk of Rain 2: Seekers of the Storm   
                          0                                                    Bombox   
True       -1             1                             The Prabbits: Happy Dogfights   
            0             1                               Payday 2: Crimewave Edition   
            1            -1                            Railway Empire 2: Bella Italia   
                          0                            River City Ransom: Underground   
                          1                                  Tetris Effect: Connected   

                                   first_release_date                platform  \
onlinecoop onlinecoopmax onlinemax                                              
False      -1            -1                2005-08-25  PC (Microsoft Windows)   
                          0                1997-09-30  PC (Microsoft Windows)   
                          1                2021-01-22                 Android   
            0            -1                2019-07-26                 Unknown   
                          0                2021-05-20    PlayStation Portable   
                          1                2024-10-24             Web browser   
            1            -1                1988-12-31                 Android   
                          0                2008-07-15                 Unknown   
True       -1             1                2025-06-05                 Unknown   
            0             1                2014-08-19                Xbox One   
            1            -1                2025-10-23                 SteamVR   
                          0                2011-08-30                Xbox 360   
                          1                2026-03-11  PC (Microsoft Windows)   

                                    game_modes_Battle Royale  \
onlinecoop onlinecoopmax onlinemax                             
False      -1            -1                              0.0   
                          0                              0.0   
                          1                              0.0   
            0            -1                              0.0   
                          0                              0.0   
                          1                              0.0   
            1            -1                              0.0   
                          0                              0.0   
True       -1             1                              0.0   
            0             1                              0.0   
            1            -1                              0.0   
                          0                              0.0   
                          1                              0.0   

                                    game_modes_Co-operative  \
onlinecoop onlinecoopmax onlinemax                            
False      -1            -1                             0.0   
                          0                             0.0   
                          1                             0.0   
            0            -1                             0.0   
                          0                             1.0   
                          1                             0.0   
            1            -1                             0.0   
 

After inspecting some examples of each combination of the `offlinecoop`, `offlinecoopmax`, and `offlinemax` columns; the meanings can be interpreted as follows:

| offlinecoop | offlinecoopmax | offlinemax | Meaning |
| -------- | ------- | -------- | -------- |
| True | -1 | 1 | Misinput; trust data from games endpoint |
| True | 0 | 1 | Used the other column, fill coop max with max |
| True | 1 | -1 | Offline coop but no PVP |
| True | 1 | 0 | Offline coop but no PVP |
| True | 1 | 1 | Offline coop and PVP |
| False | -1 | -1 | No offline play |
| False | -1 | 0 | No offline play |
| False | -1 | 1 | Offline PVP but no coop |
| False | 0 | -1 | No offline play |
| False | 0 | 0 | No offline play |
| False | 0 | 1 | Inconsistent, assume no offline play |
| False | 1 | -1 | Inconsistent; assume no offline play|
| False | 1 | 0 | Inconsistent; assume no offline play|


There are only 5 error cases:

1. offlinecoop = True, offlinecoopmax \< 0, and offlinemax \> 0 

2. offlinecoop = True, offlinecoopmax = 0, and offlinemax \> 0 

3. offlinecoop = False, offlinecoopmax = 0, and offlinemax \> 0 

4. offlinecoop = False, offlinecoopmax \> 0, and offlinemax \< 0 

5. offlinecoop = False, offlinecoopmax \> 0, and offlinemax = 0 


In [ ]:
from data_cleaning_functions import get_replacement_function

offline = {'coop_column' : 'offlinecoop',
           'coop_max_column' : 'offlinecoopmax',
           'max_column' : 'offlinemax'}
online = {'coop_column' : 'onlinecoop',
           'coop_max_column' : 'onlinecoopmax',
           'max_column' : 'onlinemax'}

fixed_df = full_df.apply(get_replacement_function(**offline), axis = 1)
fixed_df = fixed_df.apply(get_replacement_function(**online), axis = 1)

conflict_scenarios_df = fixed_df.groupby(columns)[selected_cols].agg(lambda x: x.sample(1))

#conflict_scenarios_df = conflict_scenarios_df.drop(labels = [()])
conflict_scenarios_df

name  \
onlinecoop onlinecoopmax onlinemax                             
False      0             0                          F-1 Race   
                         1              Little Racers Street   
True       1             0               Algos United: Live!   
                         1          Espionage: Mafia Evolved   

                                   first_release_date  \
onlinecoop onlinecoopmax onlinemax                      
False      0             0                 1988-12-31   
                         1                 1970-01-01   
True       1             0                 2024-06-04   
                         1                 2026-12-31   

                                                   platform  \
onlinecoop onlinecoopmax onlinemax                            
False      0             0          Sega Mega Drive/Genesis   
                         1           PC (Microsoft Windows)   
True       1             0           PC (Microsoft Windows)   
                         1           PC (Microsoft Windows)   

                                    game_modes_Battle Royale  \
onlinecoop onlinecoopmax onlinemax                             
False      0             0                               0.0   
                         1                               0.0   
True       1             0                               0.0   
                         1                               0.0   

                                    game_modes_Co-operative  \
onlinecoop onlinecoopmax onlinemax                            
False      0             0                              0.0   
                         1                              0.0   
True       1             0                              1.0   
                         1                              1.0   

                                    game_modes_Massively Multiplayer Online (MMO)  \
onlinecoop onlinecoopmax onlinemax                                                  
False      0             0                                                    0.0   
                         1                                                    0.0   
True       1             0                                                    0.0   
                         1                                                    0.0   

                                    game_modes_Multiplayer  \
onlinecoop onlinecoopmax onlinemax                           
False      0             0                             1.0   
                         1                             1.0   
True       1             0                             1.0   
                         1                             1.0   

                                    game_modes_Single player  \
onlinecoop onlinecoopmax onlinemax                             
False      0             0                               1.0   
                         1                               1.0   
True       1             0                               1.0   
                         1                               1.0   

                                    game_modes_Split screen  
onlinecoop onlinecoopmax onlinemax                           
False      0             0                              0.0  
                         1                              0.0  
True       1             0                              0.0  
                         1                              0.0

Todo:

Make function to clean conflicting columns
